# LightGBM


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import lightgbm as lgb
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"

train_path = processed_data_dir / "train_with_folds.parquet"
test_path = processed_data_dir / "test_clean.parquet"
sample_submission_path = raw_data_dir / "sample_submission.csv"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

train_df.shape, test_df.shape, sample_submission_df.shape

((699635, 25), (299844, 23), (299844, 2))

In [3]:
train_df.columns

Index(['id', 'Age', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Cleanliness',
       'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender',
       'Customer Type', 'Type of Travel', 'Class', 'satisfaction',
       'Arrival Delay Missing', 'fold'],
      dtype='str')

In [4]:
target = "satisfaction"

feature_cols = [c for c in train_df.columns
                 if c not in ["satisfaction", "id", "fold"]]

feature_cols, print("Number of features:", len(feature_cols))

Number of features: 22


(['Age',
  'Flight Distance',
  'Inflight wifi service',
  'Departure/Arrival time convenient',
  'Ease of Online booking',
  'Gate location',
  'Food and drink',
  'Online boarding',
  'Seat comfort',
  'Inflight entertainment',
  'On-board service',
  'Leg room service',
  'Baggage handling',
  'Checkin service',
  'Cleanliness',
  'Departure Delay in Minutes',
  'Arrival Delay in Minutes',
  'Gender',
  'Customer Type',
  'Type of Travel',
  'Class',
  'Arrival Delay Missing'],
 None)

In [5]:
x = train_df[feature_cols]
y = train_df["satisfaction"]

xtest = test_df[feature_cols]

In [6]:
categories = sorted(set(train_df["Class"].dropna().unique()) | set(test_df["Class"].dropna().unique()))

train_df["Class"] = pd.Categorical(train_df["Class"], categories=categories)
test_df["Class"] = pd.Categorical(test_df["Class"], categories=categories)

print(train_df["Class"].dtype)
print(test_df["Class"].dtype)
print(train_df["Class"].cat.categories)

category
category
Index(['Business', 'Eco', 'Eco Plus'], dtype='str')


In [7]:
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
fold_scores = []
best_iterations = []

parameters = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42
}

model = lgb.LGBMClassifier(**parameters)

In [ ]:
import gc

oof_dir.mkdir(parents=True, exist_ok=True)
test_predictions_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

fold_scores = []
best_iterations = []

for fold in range(5):

    model = lgb.LGBMClassifier(**parameters)

    train_mask = train_df["fold"] != fold
    value_mask = train_df["fold"] == fold

    x_train = train_df.loc[train_mask, feature_cols]
    y_train = train_df.loc[train_mask, target]

    x_value = train_df.loc[value_mask, feature_cols]
    y_value = train_df.loc[value_mask, target]


    model.fit(
        x_train,
        y_train,
        eval_set=[(x_value, y_value)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(100)
        ]
    )

    value_pred = model.predict_proba(
        x_value,
        num_iteration=model.best_iteration_
    )[:, 1]

    test_fold_pred = model.predict_proba(
        test_df[feature_cols],
        num_iteration=model.best_iteration_
    )[:, 1]

    oof_preds[value_mask] = value_pred
    test_preds += test_fold_pred / 5

    fold_auc = roc_auc_score(y_value, value_pred)

    fold_scores.append(fold_auc)
    best_iterations.append(model.best_iteration_)

    print("ROC AUC:", fold_auc)
    print("Best iteration:", model.best_iteration_)

    np.save(oof_dir / "lightgbm_partial.npy", oof_preds)
    np.save(test_predictions_dir / "lightgbm_partial.npy", test_preds)
    
    del model
    del x_train, x_value
    del y_train, y_value
    del value_pred, test_fold_pred

    gc.collect()

/Users/romanovskaiaeva/python-data-workshop/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248272, number of negative: 311436
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007536 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 599
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 22
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443574 -> initscore=-0.226669
[LightGBM] [Info] Start training from score -0.226669
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.954445	valid_0's binary_logloss: 0.247509
[200]	valid_0's auc: 0.95716	valid_0's binary_logloss: 0.232583
[300]	valid_0's auc: 0.957949	valid_0's binary_logloss: 0.230208
[400]	valid_0's auc: 0.9

/Users/romanovskaiaeva/python-data-workshop/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[100]	valid_0's auc: 0.953128	valid_0's binary_logloss: 0.24957
[200]	valid_0's auc: 0.955995	valid_0's binary_logloss: 0.234867
[300]	valid_0's auc: 0.956743	valid_0's binary_logloss: 0.232575
[400]	valid_0's auc: 0.95704	valid_0's binary_logloss: 0.231668
[500]	valid_0's auc: 0.957233	valid_0's binary_logloss: 0.231201
[600]	valid_0's auc: 0.957338	valid_0's binary_logloss: 0.230936
[700]	valid_0's auc: 0.957425	valid_0's binary_logloss: 0.230667
[800]	valid_0's auc: 0.957501	valid_0's binary_logloss: 0.230477
[900]	valid_0's auc: 0.957561	valid_0's binary_logloss: 0.230303
[1000]	valid_0's auc: 0.957621	valid_0's binary_logloss: 0.230139
[1100]	valid_0's auc: 0.957639	valid_0's binary_logloss: 0.230062
[1200]	valid_0's auc: 0.957659	valid_0's binary_logloss: 0.230023
[1300]	valid_0's auc: 0.957682	valid_0's binary_logloss: 0.229936
[1400]	valid_0's auc: 0.957727	valid_0's binary_logloss: 0.229854
[1500]	valid_0's auc: 0.957777	valid_0's binary_logloss: 0.229782
[1600]	valid_0's auc:

/Users/romanovskaiaeva/python-data-workshop/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[100]	valid_0's auc: 0.954693	valid_0's binary_logloss: 0.245627
[200]	valid_0's auc: 0.957685	valid_0's binary_logloss: 0.229983
[300]	valid_0's auc: 0.958473	valid_0's binary_logloss: 0.227778
[400]	valid_0's auc: 0.958826	valid_0's binary_logloss: 0.226992
[500]	valid_0's auc: 0.959017	valid_0's binary_logloss: 0.226498
[600]	valid_0's auc: 0.959128	valid_0's binary_logloss: 0.226206
[700]	valid_0's auc: 0.959238	valid_0's binary_logloss: 0.225978
[800]	valid_0's auc: 0.959305	valid_0's binary_logloss: 0.225739
[900]	valid_0's auc: 0.959352	valid_0's binary_logloss: 0.225593
[1000]	valid_0's auc: 0.9594	valid_0's binary_logloss: 0.225425
[1100]	valid_0's auc: 0.959471	valid_0's binary_logloss: 0.225269
[1200]	valid_0's auc: 0.959482	valid_0's binary_logloss: 0.225206
[1300]	valid_0's auc: 0.959514	valid_0's binary_logloss: 0.22513
[1400]	valid_0's auc: 0.959534	valid_0's binary_logloss: 0.22509
[1500]	valid_0's auc: 0.959558	valid_0's binary_logloss: 0.22503
[1600]	valid_0's auc: 0.

/Users/romanovskaiaeva/python-data-workshop/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[100]	valid_0's auc: 0.953884	valid_0's binary_logloss: 0.246923
[200]	valid_0's auc: 0.957076	valid_0's binary_logloss: 0.231972
[300]	valid_0's auc: 0.958006	valid_0's binary_logloss: 0.229428
[400]	valid_0's auc: 0.958309	valid_0's binary_logloss: 0.228625
[500]	valid_0's auc: 0.958474	valid_0's binary_logloss: 0.228113
[600]	valid_0's auc: 0.958531	valid_0's binary_logloss: 0.227836
[700]	valid_0's auc: 0.95859	valid_0's binary_logloss: 0.227576
[800]	valid_0's auc: 0.958645	valid_0's binary_logloss: 0.227398
[900]	valid_0's auc: 0.958695	valid_0's binary_logloss: 0.227252
[1000]	valid_0's auc: 0.958729	valid_0's binary_logloss: 0.227147
[1100]	valid_0's auc: 0.958778	valid_0's binary_logloss: 0.227024
[1200]	valid_0's auc: 0.958791	valid_0's binary_logloss: 0.226965
[1300]	valid_0's auc: 0.958799	valid_0's binary_logloss: 0.226892
Early stopping, best iteration is:
[1251]	valid_0's auc: 0.958805	valid_0's binary_logloss: 0.226926
ROC AUC: 0.958804821753064
Best iteration: 1251
[Li

/Users/romanovskaiaeva/python-data-workshop/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[100]	valid_0's auc: 0.954323	valid_0's binary_logloss: 0.247496
[200]	valid_0's auc: 0.957442	valid_0's binary_logloss: 0.231956
[300]	valid_0's auc: 0.958401	valid_0's binary_logloss: 0.229502
[400]	valid_0's auc: 0.958723	valid_0's binary_logloss: 0.228685
[500]	valid_0's auc: 0.958859	valid_0's binary_logloss: 0.228251
[600]	valid_0's auc: 0.958958	valid_0's binary_logloss: 0.227954
[700]	valid_0's auc: 0.959021	valid_0's binary_logloss: 0.227684
[800]	valid_0's auc: 0.959063	valid_0's binary_logloss: 0.227507
[900]	valid_0's auc: 0.959104	valid_0's binary_logloss: 0.227371
[1000]	valid_0's auc: 0.959133	valid_0's binary_logloss: 0.227249
Early stopping, best iteration is:
[996]	valid_0's auc: 0.959138	valid_0's binary_logloss: 0.227246
ROC AUC: 0.9591382330360952
Best iteration: 996


In [17]:
oof_auc = roc_auc_score(train_df[target], oof_preds)

np.save(oof_dir / "lightgbm.npy", oof_preds)
np.save(test_predictions_dir / "lightgbm.npy", test_preds)

metrics = {
    "parameters": parameters,
    "fold_auc": [float(score) for score in fold_scores],
    "mean_fold_auc": float(np.mean(fold_scores)),
    "oof_auc": float(oof_auc),
    "best_iterations": [int(value) for value in best_iterations],
}

with (metrics_dir / "lightgbm.json").open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

print("Fold AUC:", fold_scores)
print("Mean fold AUC:", np.mean(fold_scores))
print("OOF ROC AUC:", oof_auc)
print("Results saved")

submissions_dir = project_root / "submissions"
submissions_dir.mkdir(parents=True, exist_ok=True)

submission = sample_submission_df.copy()
submission["satisfaction"] = test_preds
submission.to_csv(submissions_dir / "lightgbm_submission.csv", index=False)

print("Submission saved")

Fold AUC: [0.9590555113061927, 0.9578511343210239, 0.9596026486923903, 0.958804821753064, 0.9591382330360952]
Mean fold AUC: 0.9588904698217533
OOF ROC AUC: 0.9588781741592428
Results saved
Submission saved
